# Train LiDAR / DG-VDSR

This notebook wires together:
- `model_anchor.py`
- `dataloader_anchor.py`
- `Hann_merge`
- training / validation / checkpoint saving
- progress bars and epoch metrics


In [1]:
import os, sys, json, math, time, gc, shutil
from pathlib import Path

import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

# ── Detect environment ────────────────────────────────────────────────────
IN_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')
IN_LOCAL  = not IN_COLAB
print(f'Environment: {"Colab" if IN_COLAB else "Local"}')

# =============================================================================
# LOCAL WORKSTATION (P4000)
# Dataset is expected at: <project_dir>/Dataset/
# Scripts are expected at: <project_dir>/  (same folder as this notebook)
# =============================================================================
if IN_LOCAL:
    PROJECT_DIR = Path(r'D:\Projects\dem-lidar')   # <-- edit if different
    DATASET_DIR = PROJECT_DIR / 'Dataset'

    if not DATASET_DIR.exists():
        raise FileNotFoundError(
            f'Dataset not found at {DATASET_DIR}.\n'
            f'Download it and place it at {DATASET_DIR}, or update PROJECT_DIR above.'
        )

    # Add project dir to path so scripts can be imported directly
    if str(PROJECT_DIR) not in sys.path:
        sys.path.insert(0, str(PROJECT_DIR))
    print(f'Dataset found: {DATASET_DIR}')
    print(f'Scripts:       {PROJECT_DIR}')

# =============================================================================
# GOOGLE COLAB (T4)
# Mounts Drive, copies dataset to fast local SSD (/content/dataset)
# =============================================================================
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_DATASET = Path('/content/drive/MyDrive/dem-lidar/Dataset')
    LOCAL_DATASET = Path('/content/dataset')

    if not LOCAL_DATASET.exists():
        if DRIVE_DATASET.exists():
            print('Copying dataset from Drive to local SSD (faster I/O)...')
            shutil.copytree(str(DRIVE_DATASET), str(LOCAL_DATASET))
            print('Done.')
        else:
            raise FileNotFoundError(
                f'Dataset not found at {DRIVE_DATASET}.\n'
                f'Upload your dataset to Google Drive at that path first.'
            )
    else:
        print(f'Local dataset already exists: {LOCAL_DATASET}')

    # Scripts: sourced from Drive (uploaded by setup_colab.ipynb)
    SCRIPT_DIR = Path('/content')
    if str(SCRIPT_DIR) not in sys.path:
        sys.path.insert(0, str(SCRIPT_DIR))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copying dataset from Google Drive to local /content/dataset...
Dataset copied!
Device: cuda


In [2]:
from pathlib import Path

# -----------------------------------------------------------------------------
# Paths
# BASE_DIR and CHECKPOINT_DIR are set automatically from Cell 1 variables.
# -----------------------------------------------------------------------------

# ── P4000 / local workstation ──────────────────────────────────────────────
# BASE_DIR and CHECKPOINT_DIR are derived from PROJECT_DIR set in Cell 1.
# No edits needed here if you set PROJECT_DIR correctly in Cell 1.
if IN_LOCAL:
    BASE_DIR       = str(DATASET_DIR)   # = PROJECT_DIR / 'Dataset'
    CHECKPOINT_DIR = PROJECT_DIR / 'checkpoints_dg_vdsr'

# ── Colab / T4 ─────────────────────────────────────────────────────────────
if IN_COLAB:
    BASE_DIR       = str(LOCAL_DATASET)  # = /content/dataset (fast local SSD)
    CHECKPOINT_DIR = Path('/content/drive/MyDrive/checkpoints_dg_vdsr')

TRAIN_DIRS = [
    f"{BASE_DIR}/Kl/tensors/train",
    f"{BASE_DIR}/SG/tensors/train",
]
VAL_DIRS = [
    f"{BASE_DIR}/Kl/tensors/validation_contiguous",
    f"{BASE_DIR}/SG/tensors/validation_contiguous",
]
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

RUN_NAME = "dg_vdsr_lidar"

# -----------------------------------------------------------------------------
# Training hyperparameters
# -----------------------------------------------------------------------------
EPOCHS = 500
BATCH_SIZE = 16          # P4000 FP32: ~1.3 GB activations; try 32 if no OOM
# BATCH_SIZE = 64        # T4 AMP (FP16 halves activation memory)
LR = 1e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0

TOPO_LAYERS = 18
FEATURES = 64

# Loss weights
LOSS_ALPHA      = 1.0      # Base elevation (optical, halo-masked)
LOSS_BETA       = 1.5      # Slope (supervised outside / smooth inside buffer)
LOSS_GAMMA      = 0.5      # Curvature (kept low: resists terrain flex the most)
LOSS_LAMBDA_PIN = 1.0      # Starting value; ramped to 5.0 over first 15 epochs
PIN_BETA        = 1.0      # SmoothL1 knee in metres (robust to +-2.48m scatter)
DECAY_RADIUS    = 15.0     # Halo decay in metres (~one ATL08 along-track spacing)
BUFFER_SIZE     = 3        # Buffer ring in PIXELS (3px x 10m = 30m ring)

# Lambda_pin warm-up schedule
LAMBDA_PIN_MAX   = 5.0
LAMBDA_PIN_START = 1.0
WARMUP_EPOCHS    = 15

# Dataloader settings
TRAIN_CROP = 128
VAL_CROP = 256
VAL_OVERLAP = 192
# P4000 workstation: 48-thread Xeon + 64 GB RAM → 8 workers is comfortable
NUM_WORKERS = 8
# NUM_WORKERS = 4        # Colab T4 (fewer CPUs available)
# NUM_WORKERS = 0        # Windows fallback if multiprocessing crashes

# Inference-only (no grads stored); peak ≈ 2x one feature map in memory
VAL_PATCH_BATCH = 32          # P4000 FP32 256×256: ~2 GB peak VRAM
# VAL_PATCH_BATCH = 128       # T4 AMP

# Training control
EARLY_STOPPING_PATIENCE = 500

# LR Scheduler
SCHEDULER_PATIENCE = 15
SCHEDULER_FACTOR   = 0.5
SCHEDULER_COOLDOWN = 2
SCHEDULER_MIN_LR   = 1e-6
BEST_METRIC_NAME = "composite"

training_config = {
    'run_name': RUN_NAME,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'lr': LR,
    'weight_decay': WEIGHT_DECAY,
    'grad_clip_norm': GRAD_CLIP_NORM,
    'topo_layers': TOPO_LAYERS,
    'features': FEATURES,
    'loss_alpha': LOSS_ALPHA,
    'loss_beta': LOSS_BETA,
    'loss_gamma': LOSS_GAMMA,
    'loss_lambda_pin': LOSS_LAMBDA_PIN,
    'lambda_pin_max': LAMBDA_PIN_MAX,
    'lambda_pin_start': LAMBDA_PIN_START,
    'warmup_epochs': WARMUP_EPOCHS,
    'pin_beta': PIN_BETA,
    'decay_radius': DECAY_RADIUS,
    'buffer_size': BUFFER_SIZE,
    'scheduler_patience': SCHEDULER_PATIENCE,
    'scheduler_factor': SCHEDULER_FACTOR,
    'scheduler_cooldown': SCHEDULER_COOLDOWN,
    'scheduler_min_lr': SCHEDULER_MIN_LR,
    'train_crop': TRAIN_CROP,
    'val_crop': VAL_CROP,
    'val_overlap': VAL_OVERLAP,
    'val_patch_batch': VAL_PATCH_BATCH,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'best_metric_name': BEST_METRIC_NAME,
    'train_dirs': TRAIN_DIRS,
    'val_dirs': VAL_DIRS,
}

print(json.dumps(training_config, indent=2))

{
  "run_name": "dg_vdsr_lidar",
  "epochs": 500,
  "batch_size": 64,
  "lr": 0.0001,
  "weight_decay": 0.0001,
  "grad_clip_norm": 1.0,
  "topo_layers": 18,
  "features": 64,
  "loss_alpha": 1.0,
  "loss_beta": 1.5,
  "loss_gamma": 0.5,
  "loss_lambda_pin": 1.0,
  "lambda_pin_max": 5.0,
  "lambda_pin_start": 1.0,
  "warmup_epochs": 15,
  "pin_beta": 1.0,
  "decay_radius": 15.0,
  "buffer_size": 3,
  "scheduler_patience": 15,
  "scheduler_factor": 0.5,
  "scheduler_cooldown": 2,
  "scheduler_min_lr": 1e-06,
  "train_crop": 128,
  "val_crop": 256,
  "val_overlap": 192,
  "val_patch_batch": 128,
  "early_stopping_patience": 500,
  "best_metric_name": "composite",
  "train_dirs": [
    "/content/dataset/Kl/tensors/train",
    "/content/dataset/SG/tensors/train"
  ],
  "val_dirs": [
    "/content/dataset/Kl/tensors/validation_contiguous",
    "/content/dataset/SG/tensors/validation_contiguous"
  ]
}


In [3]:
# -----------------------------------------------------------------------------
# Train / validation helpers
# -----------------------------------------------------------------------------
def compute_rmse(pred, gt):
    return torch.sqrt(torch.mean((pred - gt) ** 2))

def compute_psnr(pred, gt, eps=1e-8):
    mse = torch.mean((pred - gt) ** 2)
    data_range = torch.clamp(gt.max() - gt.min(), min=eps)
    if mse <= eps:
        return torch.tensor(float('inf'), device=pred.device)
    return 20.0 * torch.log10(data_range) - 10.0 * torch.log10(torch.clamp(mse, min=eps))

def safe_conv(tensor, kernel):
    padded = F.pad(tensor, (1, 1, 1, 1), mode='replicate')
    return F.conv2d(padded, kernel)

sobel_x = torch.tensor(
    [[-1, 0, 1],
     [-2, 0, 2],
     [-1, 0, 1]],
    dtype=torch.float32
).view(1, 1, 3, 3) / 8.0

sobel_y = torch.tensor(
    [[-1, -2, -1],
     [0, 0, 0],
     [1, 2, 1]],
    dtype=torch.float32
).view(1, 1, 3, 3) / 8.0

laplacian = torch.tensor(
    [[0, 1, 0],
     [1, -4, 1],
     [0, 1, 0]],
    dtype=torch.float32
).view(1, 1, 3, 3)

def save_checkpoint(
    epoch,
    model,
    optimizer,
    scheduler,
    best_metric,
    best_epoch,
    train_history,
    checkpoint_dir,
    training_config,
    is_best=False,
):
    ckpt = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'best_metric': best_metric,
        'best_epoch': best_epoch,
        'best_metric_name': training_config.get('best_metric_name', 'val_slope_rmse'),
        'train_history': train_history,
        'training_config': training_config,
        'torch_version': torch.__version__,
        'optimizer_hyperparams': {
            'lr': optimizer.param_groups[0].get('lr', None),
            'betas': optimizer.param_groups[0].get('betas', None),
            'eps': optimizer.param_groups[0].get('eps', None),
            'weight_decay': optimizer.param_groups[0].get('weight_decay', None),
        },
    }

    filename = checkpoint_dir / f"{RUN_NAME}_epoch_{epoch:03d}.pt"
    torch.save(ckpt, filename)

    meta_path = checkpoint_dir / f"{RUN_NAME}_epoch_{epoch:03d}_meta.json"
    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump({
            'epoch': epoch,
            'best_metric': best_metric,
            'best_epoch': best_epoch,
            'best_metric_name': training_config.get('best_metric_name', 'val_slope_rmse'),
            'training_config': training_config,
            'optimizer_hyperparams': ckpt['optimizer_hyperparams'],
            'torch_version': torch.__version__,
        }, f, indent=2)

    if is_best:
        best_path = checkpoint_dir / f"{RUN_NAME}_best.pt"
        torch.save(ckpt, best_path)

    return filename


def train_one_epoch(model, criterion, optimizer, train_loader, device, scaler, grad_clip_norm=1.0, epoch=0):
    model.train()

    LOG_GRAD_EPOCHS = 20  # log per-stream grad norms for first N epochs
    running = {
        'total': 0.0, 'base': 0.0, 'slope': 0.0,
        'curve': 0.0, 'pin': 0.0, 'mae': 0.0, 'rmse': 0.0, 'anchor_mae': 0.0,
        'grad_norm_a': 0.0, 'grad_norm_b': 0.0,
    }
    n_batches = 0

    pbar = tqdm(train_loader, desc='Train', leave=False, dynamic_ncols=True)

    for batch in pbar:
        # 1. Move data to GPU
        dem_bic = batch['dem_bic'].to(device, non_blocking=True, dtype=torch.float32)
        lidar_delta = batch['lidar_delta'].to(device, non_blocking=True, dtype=torch.float32)
        mask = batch['mask'].to(device, non_blocking=True, dtype=torch.float32)
        dist_map = batch['dist_map'].to(device, non_blocking=True, dtype=torch.float32)
        gt_dem = batch['gt_dem'].to(device, non_blocking=True, dtype=torch.float32)

        lidar_raw = dem_bic + lidar_delta

        optimizer.zero_grad(set_to_none=True)

        # ── FP32 forward pass (P4000: no FP16 support) ──────────────────────
        # To re-enable AMP on a capable GPU (e.g. T4), comment this block
        # and uncomment the AMP block below.
        pred_dem, alpha, r_topo, r_anchor = model(dem_bic, lidar_delta, mask, dist_map)
        loss_dict = criterion(pred_dem, gt_dem, lidar_raw, mask, dist_map)
        loss_dict['total'].backward()

        if grad_clip_norm is not None and grad_clip_norm > 0:
            # Pre-clip norms (logged for first LOG_GRAD_EPOCHS epochs to calibrate)
            if epoch <= LOG_GRAD_EPOCHS:
                grad_norm_a = torch.nn.utils.clip_grad_norm_(
                    model.stream_a.parameters(), grad_clip_norm
                )
                grad_norm_b = torch.nn.utils.clip_grad_norm_(
                    model.stream_b.parameters(), grad_clip_norm
                )
                running['grad_norm_a'] += float(grad_norm_a)
                running['grad_norm_b'] += float(grad_norm_b)
            else:
                torch.nn.utils.clip_grad_norm_(model.stream_a.parameters(), grad_clip_norm)
                torch.nn.utils.clip_grad_norm_(model.stream_b.parameters(), grad_clip_norm)
        optimizer.step()

        # ── AMP block (T4 / Ampere+): comment FP32 block above, uncomment below ──
        # with torch.amp.autocast('cuda'):
        #     pred_dem, alpha, r_topo, r_anchor = model(dem_bic, lidar_delta, mask, dist_map)
        #     loss_dict = criterion(pred_dem, gt_dem, lidar_raw, mask, dist_map)
        # scaler.scale(loss_dict['total']).backward()
        # if grad_clip_norm is not None and grad_clip_norm > 0:
        #     scaler.unscale_(optimizer)
        #     if epoch <= LOG_GRAD_EPOCHS:
        #         grad_norm_a = torch.nn.utils.clip_grad_norm_(model.stream_a.parameters(), grad_clip_norm)
        #         grad_norm_b = torch.nn.utils.clip_grad_norm_(model.stream_b.parameters(), grad_clip_norm)
        #         running['grad_norm_a'] += float(grad_norm_a)
        #         running['grad_norm_b'] += float(grad_norm_b)
        #     else:
        #         torch.nn.utils.clip_grad_norm_(model.stream_a.parameters(), grad_clip_norm)
        #         torch.nn.utils.clip_grad_norm_(model.stream_b.parameters(), grad_clip_norm)
        # scaler.step(optimizer)
        # scaler.update()

        # Metrics calculation remains in float32 for accuracy
        with torch.no_grad():
            batch_mae = F.l1_loss(pred_dem, gt_dem)
            batch_rmse = compute_rmse(pred_dem, gt_dem)

            # Calculate error strictly at the LiDAR locations
            mask_sum = mask.sum()
            if mask_sum > 0:
                batch_anchor_mae = (mask * torch.abs(pred_dem - lidar_raw)).sum() / mask_sum
            else:
                batch_anchor_mae = torch.tensor(0.0, device=device)


        running['total'] += float(loss_dict['total'].item())
        running['base'] += float(loss_dict['base'].item())
        running['slope'] += float(loss_dict['slope'].item())
        running['curve'] += float(loss_dict['curve'].item())
        running['pin'] += float(loss_dict['pin'].item())
        running['mae'] += float(batch_mae.item())
        running['rmse'] += float(batch_rmse.item())
        running['anchor_mae'] += float(batch_anchor_mae.item())
        n_batches += 1

        pbar.set_postfix({
            'loss': f"{loss_dict['total'].item():.4f}",
            'mae': f"{batch_mae.item():.4f}",
            'rmse': f"{batch_rmse.item():.4f}",
            'lr': f"{optimizer.param_groups[0]['lr']:.2e}",
        })

    for k in running:
        running[k] /= max(n_batches, 1)

    return running

@torch.inference_mode()
def validate_one_epoch(model, criterion, val_loader, device, val_patch_batch=64):
    model.eval()

    # Keep metric kernels on CPU - no need to use GPU for full-DEM convolutions
    sobel_x_cpu = sobel_x.cpu()
    sobel_y_cpu = sobel_y.cpu()
    laplacian_cpu = laplacian.cpu()

    total_loss = 0.0
    total_mae = 0.0
    total_rmse = 0.0
    total_psnr = 0.0
    total_slope_rmse = 0.0
    total_curve_rmse = 0.0
    total_anchor_mae = 0.0
    n_samples = 0

    outer = tqdm(val_loader, desc='Validate files', leave=False, dynamic_ncols=True)

    for sample_idx, batch in enumerate(outer, start=1):
        dem_bic_all = batch['dem_bic'].squeeze(0).to(device, dtype=torch.float32)
        lidar_delta_all = batch['lidar_delta'].squeeze(0).to(device, dtype=torch.float32)
        mask_all = batch['mask'].squeeze(0).to(device, dtype=torch.float32)
        dist_map_all = batch['dist_map'].squeeze(0).to(device, dtype=torch.float32)
        gt_dem_all = batch['gt_dem'].squeeze(0).to(device, dtype=torch.float32)

        patch_mean_all = batch['patch_mean'].squeeze(0)
        coords_all = batch['coords'].squeeze(0)
        canvas_shape = batch['canvas_shape'].squeeze(0).tolist()
        pad = batch["pad"].item()

        original_shape = (
            batch["original_shape"]
            .squeeze(0)
            .tolist()
        )

        # Keep gt_canvas on CPU - GPU not needed for final metric computation
        gt_canvas = (
            batch['gt_canvas_full']
            .squeeze(0)
            .float()
            .unsqueeze(0)
            .unsqueeze(0)
        )

        # Accumulate on CPU to avoid holding a large canvas tensor on GPU
        merger = HannStreamMerger(
            canvas_shape=canvas_shape,
            patch_size=256,
            device="cpu",
            pad=pad,
            original_shape=original_shape,
        )

        patch_preds = []
        patch_count = dem_bic_all.shape[0]
        n_patch_batches = math.ceil(patch_count / val_patch_batch)

        inner = tqdm(
            range(n_patch_batches),
            desc=f'Patches {sample_idx}/{len(val_loader)}',
            leave=False,
            dynamic_ncols=True
        )

        running_val_loss = {'total': 0.0}
        n_val_batches = 0
        file_anchor_mae_sum = 0.0
        file_mask_sum = mask_all.sum()

        for batch_idx in inner:
            start = batch_idx * val_patch_batch
            end = min(start + val_patch_batch, patch_count)

            dem_bic = dem_bic_all[start:end]
            lidar_delta = lidar_delta_all[start:end]
            mask = mask_all[start:end]
            dist_map = dist_map_all[start:end]
            
            with torch.amp.autocast('cuda'):
                pred_dem, alpha, r_topo, r_anchor = model(
                    dem_bic,
                    lidar_delta,
                    mask,
                    dist_map
                )
                
                # Compute loss per mini-batch to prevent VRAM explosion
                lidar_raw_batch = dem_bic + lidar_delta
                gt_dem_batch = gt_dem_all[start:end]
                loss_dict = criterion(pred_dem, gt_dem_batch, lidar_raw_batch, mask, dist_map)
                running_val_loss['total'] += float(loss_dict['total'].item())
                
                if mask.sum() > 0:
                    file_anchor_mae_sum += float((mask * torch.abs(pred_dem - lidar_raw_batch)).sum().item())
                
            n_val_batches += 1

            # Remove patch_preds.append(pred_dem) to free GPU memory immediately!

            merger.add_batch(
                pred_dem.detach().cpu(),
                patch_mean_all[start:end],
                coords_all[start:end].cpu()
            )

            inner.set_postfix({
                'batch': f'{batch_idx + 1}/{n_patch_batches}',
            })

        val_loss = {'total': running_val_loss['total'] / max(n_val_batches, 1)}
        
        if file_mask_sum > 0:
            val_anchor_mae = file_anchor_mae_sum / float(file_mask_sum.item())
        else:
            val_anchor_mae = 0.0
            
        total_anchor_mae += val_anchor_mae

        # Get merged DEM on CPU - all metric math stays off the GPU
        merged_pred = merger.get_final_dem(as_tensor=True).unsqueeze(0).unsqueeze(0)  # CPU

        # Explicit cleanup: free GPU memory BEFORE starting next file
        del dem_bic_all, lidar_delta_all, mask_all, dist_map_all, gt_dem_all
        del merger
        gc.collect()
        torch.cuda.empty_cache()

        # All metric computation on CPU (gt_canvas is already CPU)
        mae = F.l1_loss(merged_pred, gt_canvas)
        rmse = compute_rmse(merged_pred, gt_canvas)
        psnr = compute_psnr(merged_pred, gt_canvas)

        pred_dx = safe_conv(merged_pred, sobel_x_cpu)
        pred_dy = safe_conv(merged_pred, sobel_y_cpu)
        gt_dx = safe_conv(gt_canvas, sobel_x_cpu)
        gt_dy = safe_conv(gt_canvas, sobel_y_cpu)

        pred_slope = torch.sqrt(pred_dx ** 2 + pred_dy ** 2)
        gt_slope = torch.sqrt(gt_dx ** 2 + gt_dy ** 2)
        slope_rmse = compute_rmse(pred_slope, gt_slope)

        pred_curve = safe_conv(merged_pred, laplacian_cpu)
        gt_curve = safe_conv(gt_canvas, laplacian_cpu)
        curve_rmse = compute_rmse(pred_curve, gt_curve)

        del merged_pred, gt_canvas

        total_loss += val_loss['total']
        total_mae += float(mae.item())
        total_rmse += float(rmse.item())
        total_psnr += float(psnr.item())
        total_slope_rmse += float(slope_rmse.item())
        total_curve_rmse += float(curve_rmse.item())
        n_samples += 1

        outer.set_postfix({
            'mae': f'{mae.item():.3f}',
            'rmse': f'{rmse.item():.3f}',
            'psnr': f'{psnr.item():.2f}',
            'slope': f'{slope_rmse.item():.3f}',
            'curve': f'{curve_rmse.item():.3f}',
        })

    return {
        'val_total': total_loss / max(n_samples, 1),
        'val_mae': total_mae / max(n_samples, 1),
        'val_rmse': total_rmse / max(n_samples, 1),
        'val_psnr': total_psnr / max(n_samples, 1),
        'val_slope_rmse': total_slope_rmse / max(n_samples, 1),
        'val_curve_rmse': total_curve_rmse / max(n_samples, 1),
        'val_anchor_mae': total_anchor_mae / max(n_samples, 1),
    }

In [4]:
def main():
    train_loader, val_loader = create_dataloaders(
        TRAIN_DIRS,
        VAL_DIRS,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        prefetch_factor=4 if NUM_WORKERS > 0 else None,
        pin_memory=True,   # P4000 supports pinned memory; faster CPU->GPU transfer
        # pin_memory=(NUM_WORKERS > 0),  # use this if running NUM_WORKERS=0
    )

    model = DistanceGatedGeoVDSR(topo_layers=TOPO_LAYERS, features=FEATURES).to(device)
    criterion = DistanceGatedTopoLoss(
        alpha        = LOSS_ALPHA,
        beta         = LOSS_BETA,
        gamma        = LOSS_GAMMA,
        lambda_pin   = LOSS_LAMBDA_PIN,   # Will be overwritten each epoch by warm-up
        pin_beta     = PIN_BETA,
        decay_radius = DECAY_RADIUS,
        buffer_size  = BUFFER_SIZE,
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr           = LR,
        weight_decay = WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode      = 'min',
        factor    = SCHEDULER_FACTOR,
        patience  = SCHEDULER_PATIENCE,
        cooldown  = SCHEDULER_COOLDOWN,
        min_lr    = SCHEDULER_MIN_LR,
    )

    # scaler = torch.amp.GradScaler('cuda')   # Uncomment for AMP on T4/Ampere+
    scaler = torch.amp.GradScaler('cuda', enabled=False)  # disabled: FP32-only (P4000)


    # ── Checkpoint Resume ─────────────────────────────────────────────────────
    # Set RESUME_CHECKPOINT to the path of a .pt file to continue training.
    # Leave as None to start from scratch.
    RESUME_CHECKPOINT = CHECKPOINT_DIR / 'dg_vdsr_lidar_epoch_020.pt'

    start_epoch    = 1
    best_metric    = float('inf')
    best_composite = float('inf')
    best_epoch     = 0
    stale_epochs   = 0
    train_history  = []

    if RESUME_CHECKPOINT is not None:
        ckpt = torch.load(RESUME_CHECKPOINT, map_location=device, weights_only=False)  # safe: own checkpoints
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if ckpt.get('scheduler_state_dict') is not None:
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        start_epoch    = ckpt['epoch'] + 1
        best_metric    = ckpt.get('best_metric', float('inf'))
        best_composite = ckpt.get('best_metric', float('inf'))
        best_epoch     = ckpt.get('best_epoch', 0)
        train_history  = ckpt.get('train_history', [])
        print(f'Resumed from epoch {ckpt["epoch"]} | best_composite={best_composite:.4f}')
    else:
        print('Starting from scratch.')

    print(f'Train files: {len(train_loader.dataset)}')
    print(f'Val files:   {len(val_loader.dataset)}')

    epoch_bar = tqdm(range(start_epoch, EPOCHS + 1), desc='Epochs', dynamic_ncols=True)

    for epoch in epoch_bar:
        t0 = time.time()

        # Lambda_pin warm-up: ramp from LAMBDA_PIN_START → LAMBDA_PIN_MAX over WARMUP_EPOCHS
        current_lambda_pin = min(
            LAMBDA_PIN_MAX,
            LAMBDA_PIN_START + (LAMBDA_PIN_MAX - LAMBDA_PIN_START) * ((epoch - 1) / WARMUP_EPOCHS)
        )
        criterion.lambda_pin = current_lambda_pin

        train_stats = train_one_epoch(
            model=model,
            criterion=criterion,
            optimizer=optimizer,
            train_loader=train_loader,
            device=device,
            scaler=scaler,
            grad_clip_norm=GRAD_CLIP_NORM,
            epoch=epoch,
        )

        val_stats = validate_one_epoch(
            model=model,
            criterion=criterion,
            val_loader=val_loader,
            device=device,
            val_patch_batch=VAL_PATCH_BATCH,
        )

        # Combined score: LiDAR accuracy + terrain smoothness
        composite = val_stats['val_slope_rmse'] + val_stats['val_anchor_mae']
        scheduler.step(composite)

        lr_now = optimizer.param_groups[0]['lr']
        elapsed = time.time() - t0

        row = {
            'epoch': epoch,
            'lr': lr_now,
            'train_total': train_stats['total'],
            'train_base': train_stats['base'],
            'train_slope': train_stats['slope'],
            'train_curve': train_stats['curve'],
            'train_pin': train_stats['pin'],
            'train_mae': train_stats['mae'],
            'train_rmse': train_stats['rmse'],
            'val_total': val_stats['val_total'],
            'val_mae': val_stats['val_mae'],
            'val_rmse': val_stats['val_rmse'],
            'val_psnr': val_stats['val_psnr'],
            'val_slope_rmse': val_stats['val_slope_rmse'],
            'val_curve_rmse': val_stats['val_curve_rmse'],
            'val_anchor_mae': val_stats['val_anchor_mae'],
            'val_composite':  val_stats['val_slope_rmse'] + val_stats['val_anchor_mae'],
            'time_sec': elapsed,
        }
        train_history.append(row)

        is_best = composite < best_composite
        if is_best:
            best_composite = composite
            best_metric    = composite   # kept for checkpoint compat
            best_epoch     = epoch
            stale_epochs   = 0
        else:
            stale_epochs += 1

        ckpt_path = save_checkpoint(
            epoch=epoch,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            best_metric=best_metric,
            best_epoch=best_epoch,
            train_history=train_history,
            checkpoint_dir=CHECKPOINT_DIR,
            training_config=training_config,
            is_best=is_best,
        )

        epoch_bar.set_postfix({
            'train': f"{train_stats['total']:.4f}",
            'val_mae': f"{val_stats['val_mae']:.3f}",
            'val_rmse': f"{val_stats['val_rmse']:.3f}",
            'psnr': f"{val_stats['val_psnr']:.2f}",
            'best': f"{best_metric:.3f}",
            'lr': f"{lr_now:.2e}",
        })

        tqdm.write(
            f"Epoch {epoch:03d} | "
            f"λpin {current_lambda_pin:.2f} | "
            f"TLoss {train_stats['total']:.4f} "
            f"(B={train_stats['base']:.3f} S={train_stats['slope']:.3f} "
            f"C={train_stats['curve']:.3f} P={train_stats['pin']:.3f}) | "
            f"gA={train_stats['grad_norm_a']:.3f} gB={train_stats['grad_norm_b']:.3f} | "
            f"AnchorMAE {val_stats['val_anchor_mae']:.4f} | "
            f"ValMAE {val_stats['val_mae']:.4f} | "
            f"SlopeRMSE {val_stats['val_slope_rmse']:.4f} | "
            f"Composite {composite:.4f} | "
            f"LR {lr_now:.2e} | "
            f"Best {best_composite:.4f} @ ep {best_epoch} | "
            f"time {elapsed:.1f}s"
        )

        if stale_epochs >= EARLY_STOPPING_PATIENCE:
            tqdm.write(f"Early stopping triggered at epoch {epoch}.")
            break

    epoch_bar.close()
    print('Training finished.')

if __name__ == '__main__':
    main()

[TRAIN DATASET] Loaded 1649 training files.
   -> Generating 2048x2048 Spatial Noise Cache in CPU RAM...
[VAL DATASET] Loaded 6 validation files.
Resumed from epoch 20 | best_composite=2.4190
Train files: 1649
Val files:   6


Epochs:   0%|          | 0/480 [00:00<?, ?it/s]

Train:   0%|          | 0/25 [00:00<?, ?it/s]

Validate files:   0%|          | 0/6 [00:00<?, ?it/s]

Patches 1/6:   0%|          | 0/16 [00:00<?, ?it/s]

...

Train:   0%|          | 0/25 [00:00<?, ?it/s]

Validate files:   0%|          | 0/6 [00:00<?, ?it/s]

Patches 1/6:   0%|          | 0/16 [00:00<?, ?it/s]

Patches 2/6:   0%|          | 0/16 [00:00<?, ?it/s]

KeyboardInterrupt: 